In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

In [ ]:
!kaggle datasets download -d rajumavinmar/finger-print-based-blood-group-dataset
!unzip finger-print-based-blood-group-dataset.zip


Streaming output truncated to the last 5000 lines.
  inflating: dataset_blood_group/A-/cluster_1_3319.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3326.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3329.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3350.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3353.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3356.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3372.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3381.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3394.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3404.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3409.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3414.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3416.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3421.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3423.BMP  
  inflating: dataset_blood_group/A-/cluster_1_3424.BMP  
  inflating: dataset_blood_group/A-/c

In [ ]:
# Path to your dataset
dataset_dir = "/content/dataset_blood_group"

# Set image dimensions
img_height, img_width = 224, 224

# Batch size and other parameters
batch_size =32
epochs =20


In [ ]:
# Create ImageDataGenerator instances for training and validation
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)


Found 4803 images belonging to 8 classes.
Found 1197 images belonging to 8 classes.


In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,992 (42.61 MB)

 Trainable params: 11,169,992 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


151/151 ━━━━━━━━━━━━━━━━━━━━ 22s 93ms/step - accuracy: 0.2280 - loss: 2.2430 - val_accuracy: 0.7109 - val_loss: 0.8588
Epoch 2/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.6201 - loss: 1.0217 - val_accuracy: 0.8062 - val_loss: 0.6568
Epoch 3/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.7030 - loss: 0.7966 - val_accuracy: 0.8672 - val_loss: 0.3992
Epoch 4/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.7490 - loss: 0.6704 - val_accuracy: 0.8421 - val_loss: 0.4367
Epoch 5/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 9s 60ms/step - accuracy: 0.7910 - loss: 0.5654 - val_accuracy: 0.8396 - val_loss: 0.4322
Epoch 6/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.7991 - loss: 0.5310 - val_accuracy: 0.8622 - val_loss: 0.3800
Epoch 7/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.8113 - loss: 0.4943 - val_accuracy: 0.8588 - val_loss: 0.3777
Epoch 8/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - accuracy: 0.8478 - loss: 0.4099 - val_accuracy: 0.

In [ ]:
val_loss, val_acc = model.evaluate(validation_generator)
print(f"Validation accuracy: {val_acc:.4f}")


38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8708 - loss: 0.5223
Validation accuracy: 0.8672


In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
img_path ="/content/dataset_blood_group/A+/cluster_0_1001.BMP"
def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

def predict_image(model, img_path):
    img_array = load_and_preprocess_image(img_path)
    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)
    return predicted_class

# Example usage

predicted_class = predict_image(model, img_path)
print(f"Predicted Blood Group: {list(train_generator.class_indices.keys())[predicted_class[0]]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 853ms/step
Predicted Blood Group: A+
